# 02 - Data Profiling

## Objective

This notebook examines the structure and quality of the raw flight dataset before exploratory analysis and data cleaning.

The profiling process includes:

- Reviewing dataset dimensions and schema
- Measuring missing values
- Identifying duplicate records
- Summarizing numerical variables
- Reviewing key categorical variables
- Performing essential data-quality checks

The findings from this notebook will guide the exploratory analysis and data-cleaning decisions.

#### Load configuration and dataset

In [0]:
# Load the project configuration and raw flight table

from config import project_config as cfg
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

df_raw = spark.read.table(cfg.RAW_TABLE)

print(f"Table loaded: {cfg.RAW_TABLE}")

#### Dataset overview

This section reviews the size and structure of the raw flight dataset.

In [0]:
# Review dataset dimensions

row_count = df_raw.count()
column_count = len(df_raw.columns)

print(f"Total records: {row_count:,}")
print(f"Total columns: {column_count}")

In [0]:
# Review the inferred schema

df_raw.printSchema()

#### Missing-value analysis

Missing values are measured for every column to identify variables that may require removal, imputation, or special treatment during data cleaning.

In [0]:
# Calculate missing-value counts for every column

null_counts = (
    df_raw
    .select(
        [
            F.sum(
                F.col(column_name).isNull().cast("long")
            ).alias(column_name)
            for column_name in df_raw.columns
        ]
    )
    .first()
)

missing_values_data = [
    (
        field.name,
        field.dataType.simpleString(),
        int(null_counts[field.name] or 0),
        round(
            (null_counts[field.name] or 0)
            / row_count
            * 100,
            2
        ),
    )
    for field in df_raw.schema.fields
]

missing_values_df = spark.createDataFrame(
    missing_values_data,
    [
        "column_name",
        "data_type",
        "null_count",
        "null_percentage",
    ],
)

display(
    missing_values_df.orderBy(
        F.col("null_percentage").desc(),
        F.col("column_name"),
    )
)

#### Duplicate-record analysis

This section identifies fully duplicated rows that could distort flight counts and analytical results.

In [0]:
# Calculate fully duplicated records

distinct_record_count = df_raw.distinct().count()
duplicate_record_count = row_count - distinct_record_count

duplicate_percentage = round(
    duplicate_record_count / row_count * 100,
    4
)

print(f"Distinct records: {distinct_record_count:,}")
print(f"Duplicate records: {duplicate_record_count:,}")
print(f"Duplicate percentage: {duplicate_percentage}%")

#### Numerical summary

This section summarizes the distribution of numerical variables, including their central tendency, variation, ranges, and quartiles.

In [0]:
# Identify numerical columns

numeric_columns = [
    field.name
    for field in df_raw.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(f"Numerical columns: {len(numeric_columns)}")

In [0]:
# Display descriptive statistics for numerical columns

display(
    df_raw
    .select(numeric_columns)
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
    )
)

#### Key categorical variables

This section reviews the cardinality and most frequent values of the main categorical variables used in the analysis.

In [0]:
# Keep only configured categorical columns available in the dataset

categorical_columns = [
    column_name
    for column_name in cfg.KEY_CATEGORICAL_COLUMNS
    if column_name in df_raw.columns
]

categorical_summary = []

for column_name in categorical_columns:
    distinct_count = (
        df_raw
        .select(column_name)
        .distinct()
        .count()
    )

    categorical_summary.append(
        (column_name, distinct_count)
    )

categorical_summary_df = spark.createDataFrame(
    categorical_summary,
    [
        "column_name",
        "distinct_value_count",
    ],
)

display(
    categorical_summary_df.orderBy(
        F.col("distinct_value_count").desc()
    )
)

In [0]:
# Display the most frequent values for key categorical columns

for column_name in categorical_columns:
    print(f"Most frequent values for {column_name}:")

    display(
        df_raw
        .groupBy(column_name)
        .count()
        .orderBy(F.col("count").desc())
        .limit(cfg.TOP_N_RESULTS)
    )

#### Essential data-quality checks

This section verifies the valid ranges and values of the main operational variables.

In [0]:
# Perform essential range and binary-value checks

quality_checks = [
    (
        "Invalid QUARTER values",
        df_raw.filter(
            F.col(cfg.QUARTER_COLUMN).isNull()
            | ~F.col(cfg.QUARTER_COLUMN).between(1, 4)
        ).count(),
    ),
    (
        "Invalid MONTH values",
        df_raw.filter(
            F.col(cfg.MONTH_COLUMN).isNull()
            | ~F.col(cfg.MONTH_COLUMN).between(1, 12)
        ).count(),
    ),
    (
        "Invalid DAY_OF_WEEK values",
        df_raw.filter(
            F.col(cfg.DAY_OF_WEEK_COLUMN).isNull()
            | ~F.col(cfg.DAY_OF_WEEK_COLUMN).between(1, 7)
        ).count(),
    ),
    (
        "Invalid CANCELLED values",
        df_raw.filter(
            F.col(cfg.CANCELLED_COLUMN).isNull()
            | ~F.col(cfg.CANCELLED_COLUMN).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid DIVERTED values",
        df_raw.filter(
            F.col(cfg.DIVERTED_COLUMN).isNull()
            | ~F.col(cfg.DIVERTED_COLUMN).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid ARR_DEL15 values",
        df_raw.filter(
            F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN).isNotNull()
            & ~F.col(
                cfg.ARRIVAL_DELAY_FLAG_COLUMN
            ).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid DEP_DEL15 values",
        df_raw.filter(
            F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN).isNotNull()
            & ~F.col(
                cfg.DEPARTURE_DELAY_FLAG_COLUMN
            ).isin(0, 1)
        ).count(),
    ),
    (
        "Non-positive DISTANCE values",
        df_raw.filter(
            F.col(cfg.DISTANCE_COLUMN).isNull()
            | (F.col(cfg.DISTANCE_COLUMN) <= 0)
        ).count(),
    ),
]

quality_checks_df = spark.createDataFrame(
    quality_checks,
    [
        "quality_check",
        "invalid_record_count",
    ],
)

display(quality_checks_df)

#### Dataset preview

A small sample is displayed to support a final visual review of the raw records.

In [0]:
display(df_raw.limit(10))

#### Profiling completion

In [0]:
print("Data profiling completed successfully.")